In [ ]:
# =========================
# INSTALACIÓN EN COLAB
# =========================
!pip -q install pandas openpyxl scipy numpy

# =========================
# IMPORTS
# =========================
from __future__ import annotations

import calendar
import datetime as dt
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import scipy.sparse as sp
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from scipy.optimize import Bounds, LinearConstraint, milp


# =========================
# CONFIGURACIÓN PARA COLAB
# =========================

HOSPITAL = "CLINICO"   # "CLINICO" o "GIL"
YEAR = 2026

WORKERS_FILE = "/content/trabajadores.xlsx"
PRESENCES_FILE = "/content/PRESENCIAS.xlsx"
OUTPUT_FILE = "/content/GIL_cuadrante_anual_clinico_2026.xlsx"

HOLIDAYS_FILE = None
FIXED_ASSIGNMENTS_FILE = None

TIME_LIMIT_PER_MONTH = 30


# =========================
# CONSTANTES
# =========================

SHIFT_HOURS = {"M": 7, "T": 7, "N": 10}
WORK_CODES = ["M", "T", "N"]
REST_CODE = "D"

WEEKDAY_NAMES = {
    0: "LUNES",
    1: "MARTES",
    2: "MIERCOLES",
    3: "JUEVES",
    4: "VIERNES",
    5: "SABADO",
    6: "DOMINGO",
}

SUMMER_MONTHS = {6, 7, 8}
SUMMER_MIN_REST = 10


# =========================
# CONFIG
# =========================

@dataclass
class Config:
    hospital: str
    year: int
    workers_file: Path
    presences_file: Path
    output_file: Path
    holidays_file: Optional[Path] = None
    fixed_assignments_file: Optional[Path] = None
    time_limit_per_month: int = 30

    understaff_penalty: float = 150.0
    over_hours_penalty: float = 3.0
    under_hours_penalty: float = 3.0
    deviation_penalty: float = 0.2
    off_day_penalty: float = 0.1


# =========================
# UTILIDADES
# =========================

def normalize_text(value: object) -> str:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return str(value).strip().upper()


ACCENTLESS = str.maketrans({
    "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N",
    "á": "A", "é": "A", "í": "I", "ó": "O", "ú": "U", "ü": "U", "ñ": "N",
})

def normalize_key(value: object) -> str:
    return normalize_text(value).translate(ACCENTLESS)


def hospital_sheet_name(hospital: str) -> str:
    h = normalize_key(hospital)
    if "CLINICO" in h:
        return "H.CLINICO"
    if "GIL" in h:
        return "H.GIL CASARES"
    raise ValueError(f"Hospital no reconocido: {hospital}")


def worker_matches_hospital(center_value: object, hospital: str) -> bool:
    center = normalize_key(center_value)
    hosp = normalize_key(hospital)
    if "CLINICO" in hosp:
        return "CLINICO" in center
    if "GIL" in hosp:
        return "GIL" in center
    return False


def infer_category(row: pd.Series) -> str:
    puesto = normalize_key(row.get("PUESTO", ""))
    return "PEON" if "PEON" in puesto else "LIMPIADOR"


def infer_allowed_shifts(row: pd.Series) -> List[str]:
    turno = normalize_key(row.get("TURNO", ""))
    obs = normalize_key(row.get("observaciones", ""))
    category = row["categoria"]

    if "NOCHE/TARDE" in turno or "NOCHES Y TARDE" in obs:
        return ["N", "T"]
    if "NOCHE" in turno or "UNA NOCHE SI UNA NOCHE NO" in obs:
        return ["N"]
    if "CONCILIACION MANANA" in turno or "CONCILIACION" in obs:
        return ["M"]
    if "DE LUNES A VIERNES" in obs:
        return ["M"] if category != "PEON" else ["M", "T"]
    if "15 DIAS MANANA/15 DIAS TARDE" in obs or "1 MES DE MANANA/1 MES DE TARDE" in obs:
        return ["M", "T"]
    if "ROTATORIO" in turno:
        return ["M", "T"]
    if turno == "MANANA":
        return ["M"]
    if turno == "TARDE":
        return ["T"]
    if category == "PEON":
        return ["M", "T"]
    return ["M", "T"]


def preferred_shift(row: pd.Series) -> Optional[str]:
    turno = normalize_key(row.get("TURNO", ""))
    if "NOCHE" in turno:
        return "N"
    if "TARDE" in turno:
        return "T"
    if "MANANA" in turno or "CONCILIACION" in turno:
        return "M"
    return None


def parse_weekend_ratio(row: pd.Series) -> Optional[int]:
    obs = normalize_key(row.get("observaciones", ""))
    if "UN FIN DE SEMANA SI Y DOS" in obs:
        return 3
    if "UN FIN DE SEMANA SI Y UNO NO" in obs or "UN SABADO SI Y UNO NO" in obs:
        return 2
    return None


def load_holidays(path: Optional[Path]) -> set[dt.date]:
    if path is None:
        return set()
    df = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)
    if "date" not in df.columns:
        df = df.rename(columns={df.columns[0]: "date"})
    return {pd.to_datetime(x).date() for x in df["date"] if not pd.isna(x)}


def load_fixed_assignments(path: Optional[Path]) -> Dict[Tuple[str, dt.date], str]:
    if path is None:
        return {}
    df = pd.read_csv(path) if path.suffix.lower() == ".csv" else pd.read_excel(path)
    cols = {normalize_key(c): c for c in df.columns}
    required = ["TRABAJADOR", "DATE", "CODE"]
    missing = [c for c in required if c not in cols]
    if missing:
        raise ValueError(f"Faltan columnas en fixed_assignments: {missing}")
    out = {}
    for _, row in df.iterrows():
        worker = str(row[cols["TRABAJADOR"]]).strip()
        date = pd.to_datetime(row[cols["DATE"]]).date()
        code = normalize_text(row[cols["CODE"]])
        out[(worker, date)] = code
    return out


# =========================
# CARGA DE DATOS
# =========================

def load_workers(path: Path, hospital: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    df = df[df["CENTRO"].apply(lambda x: worker_matches_hospital(x, hospital))].copy()
    if df.empty:
        raise ValueError(f"No se encontraron trabajadores para {hospital}")

    df["TRABAJADOR"] = df["TRABAJADOR"].astype(str).str.strip()
    df["categoria"] = df.apply(infer_category, axis=1)
    df["allowed_shifts"] = df.apply(infer_allowed_shifts, axis=1)
    df["preferred_shift"] = df.apply(preferred_shift, axis=1)
    df["weekend_ratio"] = df.apply(parse_weekend_ratio, axis=1)
    df["annual_hours"] = pd.to_numeric(df["HORAS/AÑO"], errors="coerce").fillna(1715.0)
    df["obs_norm"] = df["observaciones"].apply(normalize_key)
    return df.reset_index(drop=True)


def load_demands(path: Path, hospital: str) -> Dict[str, Dict[Tuple[str, str], int]]:
    sheet = hospital_sheet_name(hospital)
    df = pd.read_excel(path, sheet_name=sheet, header=None)

    day_names = [normalize_key(x) for x in df.iloc[1, [0, 5, 10, 15, 20, 25, 30, 35]].tolist()]
    values = df.iloc[3].tolist()

    demands: Dict[str, Dict[Tuple[str, str], int]] = {}
    for block_idx, day_name in enumerate(day_names):
        base = block_idx * 5
        demands[day_name] = {
            ("LIMPIADOR", "M"): int(math.ceil(float(values[base] or 0))),
            ("LIMPIADOR", "T"): int(math.ceil(float(values[base + 1] or 0))),
            ("LIMPIADOR", "N"): int(math.ceil(float(values[base + 2] or 0))),
            ("PEON", "M"): int(math.ceil(float(values[base + 3] or 0))),
            ("PEON", "T"): int(math.ceil(float(values[base + 4] or 0))),
        }
    return demands


# =========================
# SOLVER MENSUAL
# =========================

class MonthlySolver:
    def __init__(
        self,
        workers: pd.DataFrame,
        dates: List[dt.date],
        demands: Dict[str, Dict[Tuple[str, str], int]],
        holidays: set[dt.date],
        fixed_assignments: Dict[Tuple[str, dt.date], str],
        config: Config,
        annual_hours_remaining: Dict[int, float],
        months_remaining_including_current: int,
        previous_last_code: Dict[int, Optional[str]],
    ):
        self.workers = workers
        self.dates = dates
        self.demands = demands
        self.holidays = holidays
        self.fixed_assignments = fixed_assignments
        self.config = config
        self.annual_hours_remaining = annual_hours_remaining
        self.months_remaining = months_remaining_including_current
        self.previous_last_code = previous_last_code

        self.var_index: Dict[Tuple, int] = {}
        self.obj: List[float] = []
        self.integrality: List[int] = []
        self.lb: List[float] = []
        self.ub: List[float] = []
        self.rows: List[Tuple[List[int], List[float]]] = []
        self.lo: List[float] = []
        self.hi: List[float] = []

    def add_var(self, key: Tuple, cost: float, integer=True, lower=0.0, upper=1.0) -> int:
        idx = len(self.obj)
        self.var_index[key] = idx
        self.obj.append(cost)
        self.integrality.append(1 if integer else 0)
        self.lb.append(lower)
        self.ub.append(upper)
        return idx

    def add_constraint(self, items: Iterable[Tuple[int, float]], lower: float, upper: float):
        idxs, coefs = zip(*items) if items else ([], [])
        self.rows.append((list(idxs), list(coefs)))
        self.lo.append(lower)
        self.hi.append(upper)

    def day_type(self, date: dt.date) -> str:
        if date in self.holidays:
            return "FESTIVOS"
        return WEEKDAY_NAMES[date.weekday()]

    def target_hours_for_worker(self, wi: int) -> float:
        remaining = max(0.0, self.annual_hours_remaining[wi])
        return remaining / max(1, self.months_remaining)

    def build(self):
        for wi, row in self.workers.iterrows():
            allowed = list(row["allowed_shifts"])
            preferred = row["preferred_shift"]

            for di, date in enumerate(self.dates):
                fixed_code = self.fixed_assignments.get((row["TRABAJADOR"], date))
                for code in allowed + [REST_CODE]:
                    lo, hi = 0.0, 1.0
                    if fixed_code is not None:
                        lo = hi = 1.0 if code == fixed_code else 0.0

                    cost = self.config.off_day_penalty if code == REST_CODE else 0.0
                    if preferred and code in WORK_CODES and code != preferred:
                        cost += self.config.deviation_penalty

                    self.add_var(("x", wi, di, code), cost, True, lo, hi)

        for di, date in enumerate(self.dates):
            dt_name = self.day_type(date)
            for category in ["LIMPIADOR", "PEON"]:
                for code in WORK_CODES:
                    need = self.demands.get(dt_name, {}).get((category, code), 0)
                    if need > 0:
                        self.add_var(
                            ("u", di, category, code),
                            self.config.understaff_penalty,
                            integer=False,
                            lower=0.0,
                            upper=float(need),
                        )

        for wi, _ in self.workers.iterrows():
            self.add_var(("over", wi), self.config.over_hours_penalty, integer=False, lower=0.0, upper=np.inf)
            self.add_var(("under", wi), self.config.under_hours_penalty, integer=False, lower=0.0, upper=np.inf)

        for wi, row in self.workers.iterrows():
            allowed = list(row["allowed_shifts"])
            for di, _ in enumerate(self.dates):
                items = [(self.var_index[("x", wi, di, code)], 1.0) for code in allowed + [REST_CODE]]
                self.add_constraint(items, 1.0, 1.0)

        for di, date in enumerate(self.dates):
            dt_name = self.day_type(date)
            for category in ["LIMPIADOR", "PEON"]:
                for code in WORK_CODES:
                    need = self.demands.get(dt_name, {}).get((category, code), 0)
                    if need <= 0:
                        continue

                    items = []
                    for wi, row in self.workers.iterrows():
                        if row["categoria"] == category and code in row["allowed_shifts"]:
                            items.append((self.var_index[("x", wi, di, code)], 1.0))
                    items.append((self.var_index[("u", di, category, code)], 1.0))
                    self.add_constraint(items, float(need), np.inf)

        for wi, row in self.workers.iterrows():
            target = self.target_hours_for_worker(wi)
            items = []
            for di, _ in enumerate(self.dates):
                for code in row["allowed_shifts"]:
                    items.append((self.var_index[("x", wi, di, code)], float(SHIFT_HOURS[code])))
            items.append((self.var_index[("over", wi)], -1.0))
            items.append((self.var_index[("under", wi)], 1.0))
            self.add_constraint(items, target, target)

        for wi, row in self.workers.iterrows():
            if "DE LUNES A VIERNES" not in row["obs_norm"]:
                continue
            for di, date in enumerate(self.dates):
                if date.weekday() >= 5:
                    for code in row["allowed_shifts"]:
                        self.add_constraint([(self.var_index[("x", wi, di, code)], 1.0)], 0.0, 0.0)

        for wi, row in self.workers.iterrows():
            if "N" not in row["allowed_shifts"]:
                continue
            if self.previous_last_code.get(wi) == "N" and self.dates:
                self.add_constraint([(self.var_index[("x", wi, 0, "N")], 1.0)], 0.0, 0.0)
            for di in range(len(self.dates) - 1):
                self.add_constraint(
                    [
                        (self.var_index[("x", wi, di, "N")], 1.0),
                        (self.var_index[("x", wi, di + 1, "N")], 1.0),
                    ],
                    -np.inf,
                    1.0,
                )

        weekend_days = [i for i, d in enumerate(self.dates) if d.weekday() >= 5]
        weekends_in_month = len({(d.isocalendar().year, d.isocalendar().week) for d in self.dates if d.weekday() >= 5})
        for wi, row in self.workers.iterrows():
            ratio = row["weekend_ratio"]
            if pd.isna(ratio) or not ratio or not weekend_days:
                continue
            max_weekend_days = math.ceil((weekends_in_month / ratio) * 2.0)
            items = []
            for di in weekend_days:
                for code in row["allowed_shifts"]:
                    items.append((self.var_index[("x", wi, di, code)], 1.0))
            self.add_constraint(items, -np.inf, float(max_weekend_days))

        month = self.dates[0].month
        if month in SUMMER_MONTHS:
            for wi, _ in self.workers.iterrows():
                items = [(self.var_index[("x", wi, di, REST_CODE)], 1.0) for di, _ in enumerate(self.dates)]
                self.add_constraint(items, float(SUMMER_MIN_REST), np.inf)

    def solve(self):
        self.build()

        n_vars = len(self.obj)
        n_rows = len(self.rows)
        data, row_ind, col_ind = [], [], []

        for r, (idxs, coefs) in enumerate(self.rows):
            data.extend(coefs)
            row_ind.extend([r] * len(idxs))
            col_ind.extend(idxs)

        A = sp.coo_array((data, (row_ind, col_ind)), shape=(n_rows, n_vars)).tocsr()

        result = milp(
            c=np.array(self.obj),
            integrality=np.array(self.integrality),
            bounds=Bounds(self.lb, self.ub),
            constraints=LinearConstraint(A, self.lo, self.hi),
            options={"time_limit": self.config.time_limit_per_month},
        )

        if result.x is None:
            raise RuntimeError(f"No se obtuvo solución: {result.message}")

        return self.extract_solution(result.x), result

    def extract_solution(self, x: np.ndarray):
        rows = []
        for wi, row in self.workers.iterrows():
            out = {
                "WORKER_IDX": wi,
                "TRABAJADOR": row["TRABAJADOR"],
                "CENTRO": row["CENTRO"],
                "TURNO_BASE": row["TURNO"],
                "PUESTO": row["PUESTO"],
                "CATEGORIA": row["categoria"],
                "HORAS_AÑO": row["annual_hours"],
            }
            for di, date in enumerate(self.dates):
                chosen = REST_CODE
                for code in list(row["allowed_shifts"]) + [REST_CODE]:
                    idx = self.var_index[("x", wi, di, code)]
                    if x[idx] > 0.5:
                        chosen = code
                        break
                out[date.isoformat()] = chosen
            rows.append(out)
        return pd.DataFrame(rows)


# =========================
# REPORTING
# =========================

def build_coverage(schedule: pd.DataFrame, demands, holidays, dates):
    records = []
    for date in dates:
        date_col = date.isoformat()
        day_type = "FESTIVOS" if date in holidays else WEEKDAY_NAMES[date.weekday()]
        for category in ["LIMPIADOR", "PEON"]:
            for code in WORK_CODES:
                need = demands.get(day_type, {}).get((category, code), 0)
                if need <= 0:
                    continue
                assigned = int(((schedule["CATEGORIA"] == category) & (schedule[date_col] == code)).sum())
                records.append({
                    "fecha": date,
                    "tipo_dia": day_type,
                    "categoria": category,
                    "turno": code,
                    "necesario": need,
                    "asignado": assigned,
                    "desviacion": assigned - need,
                    "cumple": assigned >= need,
                })
    return pd.DataFrame(records)


def build_month_hours(schedule: pd.DataFrame, target_hours_by_worker: Dict[int, float], dates: List[dt.date]) -> pd.DataFrame:
    records = []
    for _, row in schedule.iterrows():
        wi = int(row["WORKER_IDX"])
        hours = 0
        shifts = 0
        rests = 0
        for date in dates:
            code = row[date.isoformat()]
            if code in SHIFT_HOURS:
                hours += SHIFT_HOURS[code]
                shifts += 1
            elif code == REST_CODE:
                rests += 1
        records.append({
            "TRABAJADOR": row["TRABAJADOR"],
            "CATEGORIA": row["CATEGORIA"],
            "HORAS_OBJETIVO_MES": round(target_hours_by_worker[wi], 2),
            "HORAS_ASIGNADAS_MES": hours,
            "DESVIACION_HORAS": round(hours - target_hours_by_worker[wi], 2),
            "TURNOS_TRABAJADOS": shifts,
            "DESCANSOS": rests,
        })
    return pd.DataFrame(records)


def write_annual_workbook(monthly_schedules, coverage_all, monthly_hours_all, annual_hours, solve_log, config: Config):
    wb = Workbook()
    wb.remove(wb.active)

    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    weekend_fill = PatternFill("solid", fgColor="FFF2CC")
    summer_fill = PatternFill("solid", fgColor="E2F0D9")
    meta_fill = PatternFill("solid", fgColor="D9EAF7")

    for month, schedule in monthly_schedules.items():
        ws = wb.create_sheet(f"{month:02d}_{calendar.month_name[month][:3].upper()}")

        fixed_cols = ["MES", "TRABAJADOR", "CENTRO", "TURNO_BASE", "PUESTO", "CATEGORIA", "HORAS_AÑO"]
        date_cols = [c for c in schedule.columns if re.fullmatch(r"\d{4}-\d{2}-\d{2}", c)]
        cols = fixed_cols + date_cols

        for j, col in enumerate(cols, start=1):
            cell = ws.cell(row=1, column=j, value=col)
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center")

        for i, row in enumerate(schedule[cols].itertuples(index=False), start=2):
            for j, value in enumerate(row, start=1):
                ws.cell(row=i, column=j, value=value)

        for idx, col in enumerate(date_cols, start=len(fixed_cols) + 1):
            date = dt.date.fromisoformat(col)
            if date.weekday() >= 5:
                for r in range(2, ws.max_row + 1):
                    ws.cell(r, idx).fill = weekend_fill
            if month in SUMMER_MONTHS:
                for r in range(2, ws.max_row + 1):
                    if ws.cell(r, idx).value == REST_CODE:
                        ws.cell(r, idx).fill = summer_fill

        widths = {"A": 8, "B": 30, "C": 14, "D": 16, "E": 18, "F": 14, "G": 12}
        for c, w in widths.items():
            ws.column_dimensions[c].width = w
        for idx in range(8, 8 + len(date_cols)):
            ws.column_dimensions[get_column_letter(idx)].width = 12
        ws.freeze_panes = "H2"

    ws = wb.create_sheet("COBERTURA_ANUAL")
    for j, col in enumerate(coverage_all.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        c.fill = header_fill
        c.font = header_font
    for i, row in enumerate(coverage_all.itertuples(index=False), start=2):
        for j, value in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=value)
    for col in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col)].width = 16
    ws.freeze_panes = "A2"

    ws = wb.create_sheet("HORAS_MENSUALES")
    for j, col in enumerate(monthly_hours_all.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        c.fill = header_fill
        c.font = header_font
    for i, row in enumerate(monthly_hours_all.itertuples(index=False), start=2):
        for j, value in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=value)
    for col in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col)].width = 20
    ws.freeze_panes = "A2"

    ws = wb.create_sheet("HORAS_ANUALES")
    for j, col in enumerate(annual_hours.columns, start=1):
        c = ws.cell(row=1, column=j, value=col)
        c.fill = header_fill
        c.font = header_font
    for i, row in enumerate(annual_hours.itertuples(index=False), start=2):
        for j, value in enumerate(row, start=1):
            ws.cell(row=i, column=j, value=value)
    for col in range(1, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(col)].width = 22
    ws.freeze_panes = "A2"

    ws = wb.create_sheet("RESUMEN")
    summary = [
        ("hospital", config.hospital),
        ("año", config.year),
        ("regla_verano", f"mínimo {SUMMER_MIN_REST} descansos en junio/julio/agosto"),
        ("faltas_cobertura_totales", int((~coverage_all["cumple"]).sum())),
        ("desviacion_media_anual_horas", float(annual_hours["DESVIACION_AÑO"].mean())),
    ]
    for i, (k, v) in enumerate(summary, start=1):
        ws.cell(i, 1, k).fill = meta_fill
        ws.cell(i, 2, v)

    start = len(summary) + 3
    for j, col in enumerate(solve_log.columns, start=1):
        c = ws.cell(start, j, col)
        c.fill = header_fill
        c.font = header_font
    for i, row in enumerate(solve_log.itertuples(index=False), start=start + 1):
        for j, value in enumerate(row, start=1):
            ws.cell(i, j, value)
    for col in range(1, max(3, ws.max_column) + 1):
        ws.column_dimensions[get_column_letter(col)].width = 24

    config.output_file.parent.mkdir(parents=True, exist_ok=True)
    wb.save(config.output_file)


# =========================
# PIPELINE ANUAL
# =========================

def run(config: Config):
    workers = load_workers(config.workers_file, config.hospital)
    demands = load_demands(config.presences_file, config.hospital)
    holidays = load_holidays(config.holidays_file)
    fixed_assignments = load_fixed_assignments(config.fixed_assignments_file)

    annual_remaining = {int(i): float(v) for i, v in workers["annual_hours"].items()}
    previous_last_code: Dict[int, Optional[str]] = {int(i): None for i in workers.index}

    monthly_schedules = {}
    coverage_list = []
    month_hours_list = []
    solve_log_rows = []

    print(f"Trabajadores cargados: {len(workers)}")
    print(f"Hospital: {config.hospital}")

    for month in range(1, 13):
        print(f"Resolviendo mes {month}...")
        dates = [dt.date(config.year, month, d) for d in range(1, calendar.monthrange(config.year, month)[1] + 1)]
        target_hours_by_worker = {wi: annual_remaining[wi] / (13 - month) for wi in workers.index}

        solver = MonthlySolver(
            workers=workers,
            dates=dates,
            demands=demands,
            holidays=holidays,
            fixed_assignments=fixed_assignments,
            config=config,
            annual_hours_remaining=annual_remaining,
            months_remaining_including_current=(13 - month),
            previous_last_code=previous_last_code,
        )
        schedule, result = solver.solve()

        coverage = build_coverage(schedule, demands, holidays, dates)
        month_hours = build_month_hours(schedule, target_hours_by_worker, dates)

        for wi, worker_name in enumerate(workers["TRABAJADOR"]):
            assigned = float(month_hours.loc[month_hours["TRABAJADOR"] == worker_name, "HORAS_ASIGNADAS_MES"].iloc[0])
            annual_remaining[wi] = max(0.0, annual_remaining[wi] - assigned)
            previous_last_code[wi] = schedule.iloc[wi][dates[-1].isoformat()]

        schedule = schedule.drop(columns=["WORKER_IDX"])
        schedule.insert(0, "MES", month)
        coverage.insert(0, "MES", month)
        month_hours.insert(0, "MES", month)

        monthly_schedules[month] = schedule
        coverage_list.append(coverage)
        month_hours_list.append(month_hours)

        solve_log_rows.append({
            "MES": month,
            "solver_status": getattr(result, "status", None),
            "solver_message": getattr(result, "message", None),
            "objective": float(getattr(result, "fun", np.nan)),
            "faltas_cobertura": int((~coverage["cumple"]).sum()),
            "desviacion_media_horas": float(month_hours["DESVIACION_HORAS"].mean()),
        })

    coverage_all = pd.concat(coverage_list, ignore_index=True)
    monthly_hours_all = pd.concat(month_hours_list, ignore_index=True)
    annual_hours = (
        monthly_hours_all.groupby("TRABAJADOR", as_index=False)
        .agg(
            CATEGORIA=("CATEGORIA", "first"),
            HORAS_ASIGNADAS_AÑO=("HORAS_ASIGNADAS_MES", "sum"),
            DESCANSOS_AÑO=("DESCANSOS", "sum"),
        )
        .merge(
            workers[["TRABAJADOR", "annual_hours"]].rename(columns={"annual_hours": "HORAS_OBJETIVO_AÑO"}),
            on="TRABAJADOR",
            how="left",
        )
    )
    annual_hours["DESVIACION_AÑO"] = annual_hours["HORAS_ASIGNADAS_AÑO"] - annual_hours["HORAS_OBJETIVO_AÑO"]
    solve_log = pd.DataFrame(solve_log_rows)

    write_annual_workbook(
        monthly_schedules=monthly_schedules,
        coverage_all=coverage_all,
        monthly_hours_all=monthly_hours_all,
        annual_hours=annual_hours,
        solve_log=solve_log,
        config=config,
    )

    print("Terminado.")
    print(f"Excel generado en: {config.output_file}")


# =========================
# EJECUCIÓN EN COLAB
# =========================

config = Config(
    hospital=HOSPITAL,
    year=YEAR,
    workers_file=Path(WORKERS_FILE),
    presences_file=Path(PRESENCES_FILE),
    output_file=Path(OUTPUT_FILE),
    holidays_file=Path(HOLIDAYS_FILE) if HOLIDAYS_FILE else None,
    fixed_assignments_file=Path(FIXED_ASSIGNMENTS_FILE) if FIXED_ASSIGNMENTS_FILE else None,
    time_limit_per_month=TIME_LIMIT_PER_MONTH,
)

run(config)

Trabajadores cargados: 12
Hospital: GIL
Resolviendo mes 1...
Resolviendo mes 2...
Resolviendo mes 3...
Resolviendo mes 4...
Resolviendo mes 5...
Resolviendo mes 6...
Resolviendo mes 7...
Resolviendo mes 8...
Resolviendo mes 9...
Resolviendo mes 10...
Resolviendo mes 11...
Resolviendo mes 12...
Terminado.
Excel generado en: /content/GIL_cuadrante_anual_clinico_2026.xlsx


# **VALIDACIÓN**

In [ ]:
import calendar
import datetime as dt
import math
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd

# =========================================
# CONFIGURACIÓN
# =========================================

HOSPITAL = "CLINICO"   # "CLINICO" o "GIL"
YEAR = 2026

GENERATED_SCHEDULE_FILE = "/content/cuadrante_anual_clinico_2026.xlsx"
WORKERS_FILE = "/content/trabajadores.xlsx"
PRESENCES_FILE = "/content/PRESENCIAS.xlsx"

OUTPUT_REPORT_FILE = "/content/validacion_restricciones.xlsx"

VALID_CODES = {"M", "T", "N", "D"}
SHIFT_HOURS = {"M": 7, "T": 7, "N": 10}
WORK_CODES = {"M", "T", "N"}
SUMMER_MONTHS = {6, 7, 8}
SUMMER_MIN_REST = 10

WEEKDAY_NAMES = {
    0: "LUNES",
    1: "MARTES",
    2: "MIERCOLES",
    3: "JUEVES",
    4: "VIERNES",
    5: "SABADO",
    6: "DOMINGO",
}

ACCENTLESS = str.maketrans({
    "Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N",
    "á": "A", "é": "E", "í": "I", "ó": "O", "ú": "U", "ü": "U", "ñ": "N",
})


# =========================================
# UTILIDADES
# =========================================

def normalize_text(value) -> str:
    if value is None or pd.isna(value):
        return ""
    return str(value).strip().upper()

def normalize_key(value) -> str:
    return normalize_text(value).translate(ACCENTLESS)

def hospital_sheet_name(hospital: str) -> str:
    h = normalize_key(hospital)
    if "CLINICO" in h:
        return "H.CLINICO"
    if "GIL" in h:
        return "H.GIL CASARES"
    raise ValueError(f"Hospital no reconocido: {hospital}")

def worker_matches_hospital(center_value, hospital: str) -> bool:
    center = normalize_key(center_value)
    hosp = normalize_key(hospital)
    if "CLINICO" in hosp:
        return "CLINICO" in center
    if "GIL" in hosp:
        return "GIL" in center
    return False

def infer_category(row: pd.Series) -> str:
    puesto = normalize_key(row.get("PUESTO", ""))
    return "PEON" if "PEON" in puesto else "LIMPIADOR"

def parse_weekend_ratio(row: pd.Series) -> Optional[int]:
    obs = normalize_key(row.get("observaciones", ""))
    if "UN FIN DE SEMANA SI Y DOS" in obs:
        return 3
    if "UN FIN DE SEMANA SI Y UNO NO" in obs or "UN SABADO SI Y UNO NO" in obs:
        return 2
    return None

def has_monday_to_friday_rule(row: pd.Series) -> bool:
    obs = normalize_key(row.get("observaciones", ""))
    return "DE LUNES A VIERNES" in obs

def month_sheet_name(month: int) -> str:
    return f"{month:02d}_{calendar.month_name[month][:3].upper()}"

def get_date_columns(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(c))]

def get_month_dates(year: int, month: int) -> List[dt.date]:
    last_day = calendar.monthrange(year, month)[1]
    return [dt.date(year, month, d) for d in range(1, last_day + 1)]

def day_type(date: dt.date, holidays: Optional[set] = None) -> str:
    holidays = holidays or set()
    if date in holidays:
        return "FESTIVOS"
    return WEEKDAY_NAMES[date.weekday()]


# =========================================
# CARGA DE DATOS BASE
# =========================================

def load_workers(path: str, hospital: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    df = df[df["CENTRO"].apply(lambda x: worker_matches_hospital(x, hospital))].copy()
    if df.empty:
        raise ValueError(f"No hay trabajadores para {hospital}")

    df["TRABAJADOR"] = df["TRABAJADOR"].astype(str).str.strip()
    df["categoria"] = df.apply(infer_category, axis=1)
    df["weekend_ratio"] = df.apply(parse_weekend_ratio, axis=1)
    df["lv_rule"] = df.apply(has_monday_to_friday_rule, axis=1)
    df["annual_hours"] = pd.to_numeric(df["HORAS/AÑO"], errors="coerce").fillna(1715.0)

    return df[[
        "TRABAJADOR", "CENTRO", "PUESTO", "TURNO", "categoria",
        "weekend_ratio", "lv_rule", "annual_hours", "observaciones"
    ]].reset_index(drop=True)

def load_demands(path: str, hospital: str) -> Dict[str, Dict[Tuple[str, str], int]]:
    sheet = hospital_sheet_name(hospital)
    df = pd.read_excel(path, sheet_name=sheet, header=None)

    day_names = [normalize_key(x) for x in df.iloc[1, [0, 5, 10, 15, 20, 25, 30, 35]].tolist()]
    values = df.iloc[3].tolist()

    demands: Dict[str, Dict[Tuple[str, str], int]] = {}
    for block_idx, day_name in enumerate(day_names):
        base = block_idx * 5
        demands[day_name] = {
            ("LIMPIADOR", "M"): int(math.ceil(float(values[base] or 0))),
            ("LIMPIADOR", "T"): int(math.ceil(float(values[base + 1] or 0))),
            ("LIMPIADOR", "N"): int(math.ceil(float(values[base + 2] or 0))),
            ("PEON", "M"): int(math.ceil(float(values[base + 3] or 0))),
            ("PEON", "T"): int(math.ceil(float(values[base + 4] or 0))),
        }
    return demands

def load_generated_workbook(path: str, year: int) -> Dict[int, pd.DataFrame]:
    xl = pd.ExcelFile(path)
    monthly = {}
    for month in range(1, 13):
        sheet = month_sheet_name(month)
        if sheet not in xl.sheet_names:
            raise ValueError(f"Falta la hoja mensual {sheet} en el Excel generado")
        df = pd.read_excel(path, sheet_name=sheet)
        monthly[month] = df
    return monthly


# =========================================
# VALIDACIONES
# =========================================

def validate_monthly_sheet_structure(monthly_sheets: Dict[int, pd.DataFrame], year: int) -> pd.DataFrame:
    rows = []
    for month, df in monthly_sheets.items():
        date_cols = get_date_columns(df)
        expected_dates = [d.isoformat() for d in get_month_dates(year, month)]

        missing = sorted(set(expected_dates) - set(date_cols))
        extra = sorted(set(date_cols) - set(expected_dates))

        ok = (len(missing) == 0 and len(extra) == 0)
        rows.append({
            "restriccion": "estructura_columnas_mes",
            "mes": month,
            "cumple": ok,
            "detalle": f"missing={missing[:5]}, extra={extra[:5]}"
        })
    return pd.DataFrame(rows)

def validate_hospital_consistency(monthly_sheets: Dict[int, pd.DataFrame], hospital: str) -> pd.DataFrame:
    rows = []
    hosp = normalize_key(hospital)
    expected_token = "CLINICO" if "CLINICO" in hosp else "GIL"

    for month, df in monthly_sheets.items():
        if "CENTRO" not in df.columns:
            rows.append({
                "restriccion": "hospital_unico",
                "mes": month,
                "cumple": False,
                "detalle": "falta columna CENTRO"
            })
            continue

        bad = df[~df["CENTRO"].astype(str).str.upper().str.contains(expected_token, na=False)]
        rows.append({
            "restriccion": "hospital_unico",
            "mes": month,
            "cumple": bad.empty,
            "detalle": f"filas_fuera_hospital={len(bad)}"
        })
    return pd.DataFrame(rows)

def validate_valid_codes(monthly_sheets: Dict[int, pd.DataFrame], year: int) -> pd.DataFrame:
    rows = []
    for month, df in monthly_sheets.items():
        bad_cells = 0
        for col in get_date_columns(df):
            bad_cells += (~df[col].astype(str).isin(VALID_CODES)).sum()

        rows.append({
            "restriccion": "codigo_valido_por_dia",
            "mes": month,
            "cumple": bad_cells == 0,
            "detalle": f"celdas_invalidas={int(bad_cells)}"
        })
    return pd.DataFrame(rows)

def validate_unique_assignment_per_day(monthly_sheets: Dict[int, pd.DataFrame], year: int) -> pd.DataFrame:
    rows = []
    for month, df in monthly_sheets.items():
        date_cols = get_date_columns(df)
        missing_assignments = 0
        for col in date_cols:
            missing_assignments += df[col].isna().sum()

        rows.append({
            "restriccion": "una_asignacion_por_trabajador_y_dia",
            "mes": month,
            "cumple": missing_assignments == 0,
            "detalle": f"celdas_vacias={int(missing_assignments)}"
        })
    return pd.DataFrame(rows)

def validate_coverage(monthly_sheets: Dict[int, pd.DataFrame], workers_df: pd.DataFrame, demands: Dict[str, Dict[Tuple[str, str], int]], year: int) -> pd.DataFrame:
    rows = []

    worker_meta = workers_df[["TRABAJADOR", "categoria"]].copy()

    for month, sched in monthly_sheets.items():
        merged = sched.merge(worker_meta, on="TRABAJADOR", how="left", suffixes=("", "_base"))
        if "categoria_base" in merged.columns and "CATEGORIA" in merged.columns:
            merged["categoria_final"] = merged["CATEGORIA"].fillna(merged["categoria_base"])
        elif "CATEGORIA" in merged.columns:
            merged["categoria_final"] = merged["CATEGORIA"]
        else:
            merged["categoria_final"] = merged["categoria"]

        monthly_failures = 0
        failure_details = []

        for d in get_month_dates(year, month):
            col = d.isoformat()
            dt_name = day_type(d)

            for category in ["LIMPIADOR", "PEON"]:
                for code in ["M", "T", "N"]:
                    need = demands.get(dt_name, {}).get((category, code), 0)
                    if need <= 0:
                        continue

                    assigned = int(((merged["categoria_final"] == category) & (merged[col] == code)).sum())
                    if assigned < need:
                        monthly_failures += 1
                        if len(failure_details) < 5:
                            failure_details.append(f"{d} {category}-{code}: {assigned}/{need}")

        rows.append({
            "restriccion": "cobertura_minima_diaria",
            "mes": month,
            "cumple": monthly_failures == 0,
            "detalle": f"fallos={monthly_failures}; ejemplos={failure_details}"
        })
    return pd.DataFrame(rows)

def validate_lv_workers(monthly_sheets: Dict[int, pd.DataFrame], workers_df: pd.DataFrame, year: int) -> pd.DataFrame:
    rows = []
    lv_workers = set(workers_df.loc[workers_df["lv_rule"], "TRABAJADOR"].astype(str))

    for month, sched in monthly_sheets.items():
        violations = 0
        examples = []

        if not lv_workers:
            rows.append({
                "restriccion": "trabajadores_lunes_a_viernes",
                "mes": month,
                "cumple": True,
                "detalle": "sin trabajadores con esta regla"
            })
            continue

        filt = sched[sched["TRABAJADOR"].astype(str).isin(lv_workers)].copy()

        for d in get_month_dates(year, month):
            if d.weekday() < 5:
                continue
            col = d.isoformat()
            bad = filt[filt[col].isin(["M", "T", "N"])]
            violations += len(bad)
            if len(examples) < 5 and len(bad) > 0:
                examples.extend([f"{name} {d}" for name in bad["TRABAJADOR"].head(5 - len(examples))])

        rows.append({
            "restriccion": "trabajadores_lunes_a_viernes",
            "mes": month,
            "cumple": violations == 0,
            "detalle": f"violaciones={violations}; ejemplos={examples}"
        })
    return pd.DataFrame(rows)

def validate_consecutive_nights(monthly_sheets: Dict[int, pd.DataFrame], year: int) -> pd.DataFrame:
    rows = []

    prev_last_night = {}

    for month, sched in monthly_sheets.items():
        violations = 0
        examples = []

        date_cols = [d.isoformat() for d in get_month_dates(year, month)]

        for _, row in sched.iterrows():
            worker = str(row["TRABAJADOR"])

            # frontera entre meses
            if worker in prev_last_night and prev_last_night[worker] == "N" and row[date_cols[0]] == "N":
                violations += 1
                if len(examples) < 5:
                    examples.append(f"{worker}: frontera mes anterior -> {date_cols[0]}")

            # dentro del mes
            for i in range(len(date_cols) - 1):
                if row[date_cols[i]] == "N" and row[date_cols[i + 1]] == "N":
                    violations += 1
                    if len(examples) < 5:
                        examples.append(f"{worker}: {date_cols[i]} y {date_cols[i+1]}")

            prev_last_night[worker] = row[date_cols[-1]]

        rows.append({
            "restriccion": "noches_no_consecutivas",
            "mes": month,
            "cumple": violations == 0,
            "detalle": f"violaciones={violations}; ejemplos={examples}"
        })
    return pd.DataFrame(rows)

def validate_summer_rest(monthly_sheets: Dict[int, pd.DataFrame], year: int) -> pd.DataFrame:
    rows = []
    for month, sched in monthly_sheets.items():
        if month not in SUMMER_MONTHS:
            continue

        violations = 0
        examples = []
        date_cols = [d.isoformat() for d in get_month_dates(year, month)]

        for _, row in sched.iterrows():
            rests = sum(1 for c in date_cols if row[c] == "D")
            if rests < SUMMER_MIN_REST:
                violations += 1
                if len(examples) < 5:
                    examples.append(f"{row['TRABAJADOR']}: {rests}")

        rows.append({
            "restriccion": "minimo_10_descansos_verano",
            "mes": month,
            "cumple": violations == 0,
            "detalle": f"violaciones={violations}; ejemplos={examples}"
        })
    return pd.DataFrame(rows)

def validate_monthly_hours_uniformity(monthly_sheets: Dict[int, pd.DataFrame], workers_df: pd.DataFrame, year: int, tolerance_hours: float = 35.0) -> pd.DataFrame:
    """
    Comprueba uniformidad aproximada.
    No es restricción dura exacta del solver, sino una validación razonable.
    """
    records = []

    annual_targets = dict(zip(workers_df["TRABAJADOR"].astype(str), workers_df["annual_hours"]))

    for month, sched in monthly_sheets.items():
        date_cols = [d.isoformat() for d in get_month_dates(year, month)]
        months_remaining = 13 - month

        violations = 0
        examples = []

        for _, row in sched.iterrows():
            worker = str(row["TRABAJADOR"])
            annual_target = float(annual_targets.get(worker, 1715.0))
            target_month = annual_target / 12.0

            assigned = 0.0
            for c in date_cols:
                if row[c] in SHIFT_HOURS:
                    assigned += SHIFT_HOURS[row[c]]

            if abs(assigned - target_month) > tolerance_hours:
                violations += 1
                if len(examples) < 5:
                    examples.append(f"{worker}: {assigned:.1f} vs {target_month:.1f}")

        records.append({
            "restriccion": "uniformidad_mensual_aproximada_horas",
            "mes": month,
            "cumple": violations == 0,
            "detalle": f"violaciones={violations}; tolerancia={tolerance_hours}; ejemplos={examples}"
        })

    return pd.DataFrame(records)

def validate_annual_hours(monthly_sheets: Dict[int, pd.DataFrame], workers_df: pd.DataFrame, year: int, tolerance_hours: float = 60.0) -> pd.DataFrame:
    rows = []

    all_records = []
    for month, sched in monthly_sheets.items():
        date_cols = [d.isoformat() for d in get_month_dates(year, month)]
        for _, row in sched.iterrows():
            hours = 0.0
            for c in date_cols:
                if row[c] in SHIFT_HOURS:
                    hours += SHIFT_HOURS[row[c]]
            all_records.append({"TRABAJADOR": str(row["TRABAJADOR"]), "MES": month, "HORAS": hours})

    month_hours = pd.DataFrame(all_records)
    annual = month_hours.groupby("TRABAJADOR", as_index=False)["HORAS"].sum()
    annual = annual.merge(
        workers_df[["TRABAJADOR", "annual_hours"]],
        left_on="TRABAJADOR",
        right_on="TRABAJADOR",
        how="left"
    )
    annual["desviacion"] = annual["HORAS"] - annual["annual_hours"]

    bad = annual[annual["desviacion"].abs() > tolerance_hours].copy()
    rows.append({
        "restriccion": "horas_anuales_objetivo_aproximadas",
        "mes": "ANUAL",
        "cumple": bad.empty,
        "detalle": f"violaciones={len(bad)}; tolerancia={tolerance_hours}; ejemplos={bad[['TRABAJADOR','HORAS','annual_hours','desviacion']].head(5).to_dict(orient='records')}"
    })

    return pd.DataFrame(rows), annual

def validate_weekend_patterns(monthly_sheets: Dict[int, pd.DataFrame], workers_df: pd.DataFrame, year: int) -> pd.DataFrame:
    """
    Valida la misma aproximación del solver:
    ratio=2 -> un finde sí / uno no
    ratio=3 -> un finde sí / dos no
    """
    rows = []
    meta = workers_df[["TRABAJADOR", "weekend_ratio"]].copy()

    for month, sched in monthly_sheets.items():
        sched = sched.merge(meta, on="TRABAJADOR", how="left")
        weekend_days = [d.isoformat() for d in get_month_dates(year, month) if d.weekday() >= 5]
        weekends_in_month = len({
            (d.isocalendar().year, d.isocalendar().week)
            for d in get_month_dates(year, month) if d.weekday() >= 5
        })

        violations = 0
        examples = []

        for _, row in sched.iterrows():
            ratio = row["weekend_ratio"]
            if pd.isna(ratio) or not ratio:
                continue

            max_weekend_days = math.ceil((weekends_in_month / ratio) * 2.0)
            worked_weekend_days = sum(1 for c in weekend_days if row[c] in WORK_CODES)

            if worked_weekend_days > max_weekend_days:
                violations += 1
                if len(examples) < 5:
                    examples.append(f"{row['TRABAJADOR']}: {worked_weekend_days}>{max_weekend_days}")

        rows.append({
            "restriccion": "patron_aproximado_fines_de_semana",
            "mes": month,
            "cumple": violations == 0,
            "detalle": f"violaciones={violations}; ejemplos={examples}"
        })

    return pd.DataFrame(rows)


# =========================================
# EJECUCIÓN
# =========================================

def main():
    workers_df = load_workers(WORKERS_FILE, HOSPITAL)
    demands = load_demands(PRESENCES_FILE, HOSPITAL)
    monthly_sheets = load_generated_workbook(GENERATED_SCHEDULE_FILE, YEAR)

    reports = []

    reports.append(validate_monthly_sheet_structure(monthly_sheets, YEAR))
    reports.append(validate_hospital_consistency(monthly_sheets, HOSPITAL))
    reports.append(validate_valid_codes(monthly_sheets, YEAR))
    reports.append(validate_unique_assignment_per_day(monthly_sheets, YEAR))
    reports.append(validate_coverage(monthly_sheets, workers_df, demands, YEAR))
    reports.append(validate_lv_workers(monthly_sheets, workers_df, YEAR))
    reports.append(validate_consecutive_nights(monthly_sheets, YEAR))
    reports.append(validate_summer_rest(monthly_sheets, YEAR))
    reports.append(validate_monthly_hours_uniformity(monthly_sheets, workers_df, YEAR, tolerance_hours=35.0))
    annual_report, annual_hours_detail = validate_annual_hours(monthly_sheets, workers_df, YEAR, tolerance_hours=60.0)
    reports.append(annual_report)
    reports.append(validate_weekend_patterns(monthly_sheets, workers_df, YEAR))

    validation = pd.concat(reports, ignore_index=True)

    resumen = (
        validation.groupby("restriccion", as_index=False)
        .agg(
            chequeos=("cumple", "count"),
            chequeos_ok=("cumple", "sum"),
        )
    )
    resumen["porcentaje_ok"] = (100.0 * resumen["chequeos_ok"] / resumen["chequeos"]).round(2)

    print("\n=== RESUMEN VALIDACIÓN ===")
    print(resumen.sort_values("porcentaje_ok"))
    print("\n=== INCUMPLIMIENTOS ===")
    print(validation[~validation["cumple"]][["restriccion", "mes", "detalle"]].head(50))

    with pd.ExcelWriter(OUTPUT_REPORT_FILE, engine="openpyxl") as writer:
        validation.to_excel(writer, sheet_name="VALIDACION", index=False)
        resumen.to_excel(writer, sheet_name="RESUMEN", index=False)
        annual_hours_detail.to_excel(writer, sheet_name="HORAS_ANUALES_DETALLE", index=False)
        workers_df.to_excel(writer, sheet_name="WORKERS_BASE", index=False)

    print(f"\nInforme guardado en: {OUTPUT_REPORT_FILE}")


main()


=== RESUMEN VALIDACIÓN ===
                             restriccion  chequeos  chequeos_ok  porcentaje_ok
10  uniformidad_mensual_aproximada_horas        12           11          91.67
0                cobertura_minima_diaria        12           12         100.00
2                estructura_columnas_mes        12           12         100.00
1                  codigo_valido_por_dia        12           12         100.00
4                         hospital_unico        12           12         100.00
5             minimo_10_descansos_verano         3            3         100.00
6                 noches_no_consecutivas        12           12         100.00
3     horas_anuales_objetivo_aproximadas         1            1         100.00
7      patron_aproximado_fines_de_semana        12           12         100.00
8           trabajadores_lunes_a_viernes        12           12         100.00
9    una_asignacion_por_trabajador_y_dia        12           12         100.00

=== INCUMPLIMIENTOS ===